# Silver Layer — ICU d_items

## Source -> Target
`<source_schema>.d_items` -> `<target_schema>.d_items`
(defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`; set with the notebook widgets).

## Grain
One row per ICU dictionary item, identified by `itemid` (official MIMIC-IV icu `d_items`;
`linksto` names the event table that uses the item). Uniqueness of `itemid` is asserted.

## Silver treatment
- Cast `itemid` to INT and `lownormalvalue` / `highnormalvalue` to DOUBLE; all other fields unchanged.
- All rows preserved; no flags, no normalisation, no deduplication.
- Profiled counts (4,095 rows and zero-defect checks) are enforced only when `enforce_profiled_counts` is true.

## Dependencies
Reads only Bronze `d_items`. It is read by Silver `datetimeevents` and `outputevents`
(dictionary integrity checks) and by Gold.

## Publication
The table is replaced with CREATE OR REPLACE semantics; downstream Gold depends on it.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

In [0]:
# ============================================================
# ICU SILVER — D_ITEMS
# STEP 1: SOURCE PROFILING & QUALITY CONTRACT
# READ-ONLY
# ============================================================

from pyspark.sql import functions as F

SOURCE = f"{BRONZE}.d_items"
TARGET = f"{SILVER}.d_items"

bronze_items = spark.table(SOURCE)

print("SOURCE:", SOURCE)
print("TARGET:", TARGET)
print("Target exists:", spark.catalog.tableExists(TARGET))

bronze_items.printSchema()

# Safe candidate conversion for the official item identifier
itemid_as_int = F.expr(
    "TRY_CAST(TRIM(CAST(itemid AS STRING)) AS INT)"
)

# Single aggregated quality profile
item_profile = bronze_items.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("itemid").alias("unique_source_ids"),

    F.countDistinct(itemid_as_int).alias("unique_integer_ids"),

    F.count_if(
        F.col("itemid").isNull()
    ).alias("null_itemids"),

    F.count_if(
        F.col("itemid").isNotNull()
        & itemid_as_int.isNull()
    ).alias("invalid_integer_ids"),

    F.count_if(
        F.col("label").isNull()
    ).alias("null_labels"),

    F.count_if(
        F.col("label").isNotNull()
        & (F.trim("label") == "")
    ).alias("empty_labels"),

    F.count_if(
        F.col("label").isNotNull()
        & (F.col("label") != F.trim("label"))
    ).alias("label_whitespace_issues"),

    F.count_if(
        F.col("linksto").isNull()
    ).alias("null_linksto")
)

display(item_profile)

# Inspect event-table references
display(
    bronze_items
    .groupBy("linksto")
    .count()
    .orderBy(F.desc("count"))
)

print("STEP 1 COMPLETE — No data modified.")


SOURCE: cdr_mimiciv.bronze.d_items
TARGET: cdr_mimiciv.silver.d_items
Target exists: True
root
 |-- itemid: integer (nullable = true)
 |-- label: string (nullable = true)
 |-- abbreviation: string (nullable = true)
 |-- linksto: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unitname: string (nullable = true)
 |-- param_type: string (nullable = true)
 |-- lownormalvalue: integer (nullable = true)
 |-- highnormalvalue: double (nullable = true)



total_rows,unique_source_ids,unique_integer_ids,null_itemids,invalid_integer_ids,null_labels,empty_labels,label_whitespace_issues,null_linksto
4095,4095,4095,0,0,0,0,0,0


linksto,count
chartevents,3055
inputevents,476
datetimeevents,190
procedureevents,173
ingredientevents,124
outputevents,77


STEP 1 COMPLETE — No data modified.


In [0]:

# ============================================================
# ICU SILVER — D_ITEMS
# STEP 2: TYPED TRANSFORMATION + QUALITY CONTRACT
# READ-ONLY — NO TABLE MODIFICATION
# ============================================================

from pyspark.sql import functions as F

# Typed expressions
itemid_int = F.expr(
    "TRY_CAST(TRIM(CAST(itemid AS STRING)) AS INT)"
)

low_value = F.expr(
    "TRY_CAST(TRIM(CAST(lownormalvalue AS STRING)) AS DOUBLE)"
)

high_value = F.expr(
    "TRY_CAST(TRIM(CAST(highnormalvalue AS STRING)) AS DOUBLE)"
)

# 1. Prepare Silver candidate
items_silver = bronze_items.select(
    itemid_int.alias("itemid"),
    F.col("label"),
    F.col("abbreviation"),
    F.col("linksto"),
    F.col("category"),
    F.col("unitname"),
    F.col("param_type"),
    low_value.alias("lownormalvalue"),
    high_value.alias("highnormalvalue")
)

# 2. Single-pass source quality audit
items_quality = bronze_items.agg(
    F.count("*").alias("source_rows"),

    F.min(itemid_int).alias("min_itemid"),

    F.count_if(
        F.col("lownormalvalue").isNotNull()
        & low_value.isNull()
    ).alias("invalid_low_values"),

    F.count_if(
        F.col("highnormalvalue").isNotNull()
        & high_value.isNull()
    ).alias("invalid_high_values"),

    F.count_if(
        low_value.isNotNull()
        & high_value.isNotNull()
        & (low_value > high_value)
    ).alias("inverted_reference_ranges"),

    F.count_if(
        F.col("abbreviation").isNotNull()
        & (F.trim("abbreviation") == "")
    ).alias("empty_abbreviations"),

    F.count_if(
        F.col("unitname").isNotNull()
        & (F.trim("unitname") == "")
    ).alias("empty_units")
)

# 3. Display the full audit
display(items_quality)

# 4. Inspect prepared schema
print("=== SILVER D_ITEMS SCHEMA ===")
items_silver.printSchema()

print("STEP 2 COMPLETE — NOT PUBLISHED")


source_rows,min_itemid,invalid_low_values,invalid_high_values,inverted_reference_ranges,empty_abbreviations,empty_units
4095,220001,0,0,0,0,0


=== SILVER D_ITEMS SCHEMA ===
root
 |-- itemid: integer (nullable = true)
 |-- label: string (nullable = true)
 |-- abbreviation: string (nullable = true)
 |-- linksto: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unitname: string (nullable = true)
 |-- param_type: string (nullable = true)
 |-- lownormalvalue: double (nullable = true)
 |-- highnormalvalue: double (nullable = true)

STEP 2 COMPLETE — NOT PUBLISHED


In [0]:

# ============================================================
# ICU SILVER — D_ITEMS
# STEP 3: VALIDATION-GATED DELTA PUBLICATION
# ============================================================

from pyspark.sql import functions as F

TARGET = f"{SILVER}.d_items"
EXPECTED_ROWS = 4095

# 1. Retrieve source profiling and transformation checks
profile = item_profile.first().asDict()
quality = items_quality.first().asDict()

# 2. Key integrity (unconditional): itemid is the grain key
assert profile["total_rows"] == quality["source_rows"], "Row count mismatch between profiling passes"
assert profile["null_itemids"] == 0, "NULL itemid"
assert profile["invalid_integer_ids"] == 0, "itemid not convertible to INT"
assert profile["unique_source_ids"] == profile["total_rows"], "itemid is not unique"
assert profile["unique_integer_ids"] == profile["total_rows"], "itemid not unique after INT cast"

# 3. Profiled expectations (enforced only when enforce_profiled_counts is true)
if ENFORCE_PROFILED_COUNTS:
    assert profile["total_rows"] == EXPECTED_ROWS

    required_zero_checks = {
        "null_labels": profile["null_labels"],
        "empty_labels": profile["empty_labels"],
        "label_whitespace_issues": profile["label_whitespace_issues"],
        "null_linksto": profile["null_linksto"],
        "invalid_low_values": quality["invalid_low_values"],
        "invalid_high_values": quality["invalid_high_values"],
        "inverted_reference_ranges": quality["inverted_reference_ranges"],
        "empty_abbreviations": quality["empty_abbreviations"],
        "empty_units": quality["empty_units"]
    }

    failures = {
        name: value
        for name, value in required_zero_checks.items()
        if value != 0
    }

    assert not failures, f"QUALITY GATE FAILED: {failures}"
    assert quality["min_itemid"] > 220000

# 4. Enforce expected Silver schema
expected_schema = {
    "itemid": "int",
    "label": "string",
    "abbreviation": "string",
    "linksto": "string",
    "category": "string",
    "unitname": "string",
    "param_type": "string",
    "lownormalvalue": "double",
    "highnormalvalue": "double"
}

actual_schema = {
    field.name: field.dataType.simpleString()
    for field in items_silver.schema.fields
}

assert list(actual_schema.items()) == list(expected_schema.items()), (
    f"SCHEMA CONTRACT FAILED: {actual_schema}"
)

# 4. Add Silver processing metadata
items_final = items_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)

print("All quality gates passed.")
print("Publishing:", TARGET)

# 5. Create or replace the Delta table (downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    items_final.writeTo(TARGET)
    .using("delta")
    .createOrReplace()
)

print("SUCCESS: Silver d_items table published (CREATE OR REPLACE).")


All quality gates passed.
Publishing: cdr_mimiciv.silver.d_items
SUCCESS: Silver d_items table published (CREATE OR REPLACE).


In [0]:

# ============================================================
# ICU SILVER — D_ITEMS
# STEP 4: FINAL PERSISTED-TABLE AUDIT
# READ-ONLY — NO DATA MODIFICATION
# ============================================================

from pyspark.sql import functions as F

TARGET = f"{SILVER}.d_items"
EXPECTED_ROWS = 4095

saved_items = spark.table(TARGET)

# 1. Enforce saved-table schema (ordered)
expected_schema = {
    "itemid": "int",
    "label": "string",
    "abbreviation": "string",
    "linksto": "string",
    "category": "string",
    "unitname": "string",
    "param_type": "string",
    "lownormalvalue": "double",
    "highnormalvalue": "double",
    "_silver_processed_at": "timestamp"
}

actual_schema = {
    field.name: field.dataType.simpleString()
    for field in saved_items.schema.fields
}

assert list(actual_schema.items()) == list(expected_schema.items()), (
    f"Schema mismatch: {actual_schema}"
)

# 2. Audit published records
audit = saved_items.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct("itemid").alias("unique_itemids"),
    F.count_if(F.col("itemid").isNull())
        .alias("null_itemids"),
    F.count_if(F.col("_silver_processed_at").isNull())
        .alias("missing_processing_timestamps")
).first().asDict()

# 3. Compare the nine dictionary fields in both directions, including
# duplicate-row counts (small reference table, so exceptAll is affordable)
cols = items_silver.columns

missing = (
    items_silver.select(*cols)
    .exceptAll(saved_items.select(*cols))
    .count()
)

unexpected = (
    saved_items.select(*cols)
    .exceptAll(items_silver.select(*cols))
    .count()
)

# 4. Enforce final quality conditions
assert audit["silver_rows"] == profile["total_rows"], "Silver rows differ from Bronze rows"
assert audit["unique_itemids"] == audit["silver_rows"], "itemid not unique in Silver"
assert audit["null_itemids"] == 0
assert audit["missing_processing_timestamps"] == 0
assert missing == 0
assert unexpected == 0

if ENFORCE_PROFILED_COUNTS:
    assert audit["silver_rows"] == EXPECTED_ROWS

print("=== FINAL D_ITEMS SILVER AUDIT ===")
print(audit)
print("Missing from saved:", missing)
print("Unexpected in saved:", unexpected)
print("Schema verification: PASSED")
print("FINAL STATUS: PASSED")

=== FINAL D_ITEMS SILVER AUDIT ===
{'silver_rows': 4095, 'unique_itemids': 4095, 'null_itemids': 0, 'missing_processing_timestamps': 0}
Missing from saved: 0
Unexpected in saved: 0
Schema verification: PASSED
FINAL STATUS: PASSED


In [0]:
dbutils.notebook.exit("d_items:OK")